# Generalized Linear Models

A GLM separates random noise, a linear predictor, and a link function.

Generalized Linear Models make the response distribution and link function explicit. That separation is useful only when the score being optimized includes the lesson's cost and validation gap, not just an attractive raw fit.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def lesson_score(losses, cost, alternative):
    losses = np.asarray(losses, dtype=float)
    raw = round(float(losses.mean()), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    return {
        "losses": losses,
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
    }


def binary_logistic_train(x_tr, y_tr, lr=0.2, steps=900, l2=0.02):
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    weights = np.zeros(X.shape[1])
    y = y_tr.astype(float)
    for step in range(steps):
        logits = X @ weights
        probs = 1.0 / (1.0 + np.exp(-logits))
        grad = X.T @ (probs - y) / y.size
        grad[1:] = grad[1:] + l2 * weights[1:]
        weights = weights - lr * grad
    return weights


def binary_logistic_predict(weights, x_te):
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    probs = 1.0 / (1.0 + np.exp(-(X @ weights)))
    return (probs >= 0.5).astype(int)


def softmax_train(x_tr, y_tr, lr=0.15, steps=1100, l2=0.01):
    classes = np.unique(y_tr)
    mapping = {label: idx for idx, label in enumerate(classes)}
    y_idx = np.array([mapping[label] for label in y_tr])
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    W = np.zeros((X.shape[1], classes.size))
    Y = np.eye(classes.size)[y_idx]
    for step in range(steps):
        logits = X @ W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(logits)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        grad = X.T @ (probs - Y) / X.shape[0]
        grad[1:, :] = grad[1:, :] + l2 * W[1:, :]
        W = W - lr * grad
    return W, classes


def softmax_predict(model, x_te):
    W, classes = model
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    logits = X @ W
    return classes[np.argmax(logits, axis=1)]


def glm_predict(x_tr, y_tr, x_te):
    classes = np.unique(y_tr)
    if classes.size == 2:
        weights = binary_logistic_train(x_tr, y_tr)
        return binary_logistic_predict(weights, x_te)
    model = softmax_train(x_tr, y_tr)
    return softmax_predict(model, x_te)


def lda_qda_predict(x_tr, y_tr, x_te, mode="lda"):
    classes, counts = np.unique(y_tr, return_counts=True)
    if x_tr.shape[0] <= classes.size or counts.min() < 2:
        model = GaussianNB(var_smoothing=1e-8)
    elif mode == "qda":
        model = QuadraticDiscriminantAnalysis(reg_param=0.08)
    else:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def gda_predict(x_tr, y_tr, x_te):
    model = GaussianNB(var_smoothing=1e-8)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def sklearn_logistic_predict(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2500)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def classifier_metrics(rungs, predictor):
    rows = []
    for level, item in enumerate(rungs, start=1):
        name, X, y = item
        accuracy = clf_accuracy(predictor, X, y)
        rows.append({"level": level, "name": name, "accuracy": float(accuracy)})
    return rows


def plot_classifier_summary(rungs, rows, predictor):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, item, row in zip(axes[:5], rungs, rows):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        preds = predictor(x_tr_s, y_tr, x_te_s)
        ax.scatter(x_te_s[:, 0], x_te_s[:, 1], c=preds, s=14, cmap="viridis", alpha=0.8)
        ax.set_title(f"D{row['level']} acc={row['accuracy']:.2f}")
        ax.set_xlabel("feature 0")
        ax.set_ylabel("feature 1")
    axes[5].plot([row["level"] for row in rows], [row["accuracy"] for row in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_title("Accuracy vs ladder rung")
    axes[5].set_xlabel("D1 to D5")
    axes[5].set_ylabel("held-out accuracy")
    plt.tight_layout()
    plt.show()

def logistic_regression_method(losses=None, cost=0.070, alternative=0.394):
    if losses is None:
        losses = np.array([0.235, 0.109, 0.471])
    return lesson_score(losses, cost, alternative)


def softmax_multinomial_regression_method(losses=None, cost=0.080, alternative=0.401):
    if losses is None:
        losses = np.array([0.246, 0.122, 0.488])
    return lesson_score(losses, cost, alternative)


def generalized_linear_models_method(losses=None, cost=0.090, alternative=0.429):
    if losses is None:
        losses = np.array([0.257, 0.135, 0.505])
    return lesson_score(losses, cost, alternative)


def linear_quadratic_discriminant_analysis_method(losses=None, cost=0.100, alternative=0.457):
    if losses is None:
        losses = np.array([0.268, 0.148, 0.522])
    return lesson_score(losses, cost, alternative)


def gaussian_discriminant_analysis_method(losses=None, cost=0.050, alternative=0.361):
    if losses is None:
        losses = np.array([0.180, 0.070, 0.539])
    return lesson_score(losses, cost, alternative)

## The concept, built once (D1)

The lesson formula is

$$g(\mathbb E[Y\mid X])=X\beta$$

For D1, the verified per-example losses are 0.257, 0.135, 0.505. The empirical risk is the average, and the model-selection score is that raw term plus the lesson cost.

In [ ]:
result = generalized_linear_models_method()
print(result)
assert np.isclose(result["raw"], 0.299)
assert np.isclose(result["score"], 0.389)
assert np.isclose(result["gap"], 0.040)

The exact arithmetic is $R_S=(0.257, 0.135, 0.505)/3=0.299$, then $score=R_S+0.090=0.389$. The tempting alternative is 0.429, so the validation gap is $0.429-0.389=0.040$.

In [ ]:
stable_score = 0.80 * result["score"]
relative_gap = result["gap"] / result["alternative"]
print(f"stable={stable_score:.3f} relative_gap={relative_gap:.3f}")
assert stable_score < result["score"]
assert relative_gap > 0.0

## The dataset ladder

The same method now runs on D1 through D5. The printed preview shows shape, class balance or target scale, and a small sample before any fitting.

In [ ]:
rungs = clf_ladder()
for level, item in enumerate(rungs, start=1):
    name, X, y = item
    labels, counts = np.unique(y, return_counts=True)
    class_info = dict(zip(labels.tolist(), counts.tolist()))
    print(f"D{level}: {name} X={X.shape} classes={class_info}")
    print("sample X", np.round(X[:3, : min(3, X.shape[1])], 3))
    print("sample y", y[:3])

## Run the same method across D1-D5

The single headline metric for this lesson is accuracy. Classification topics also print the shared logistic baseline for a no-special-skill comparison.

In [ ]:
predictor = glm_predict
rows = classifier_metrics(rungs, predictor)
print("rung | accuracy | logistic baseline | dataset")
for row, item in zip(rows, rungs):
    name, X, y = item
    baseline = clf_accuracy(logistic_baseline, X, y)
    print(f"D{row['level']} | {row['accuracy']:.3f} | {baseline:.3f} | {row['name']}")
assert len(rows) == 5
assert all(0.0 <= row["accuracy"] <= 1.0 for row in rows)

## Results visualization

The closing figure has two parts: small multiples for each rung and one summary curve from D1 to D5.

In [ ]:
plot_classifier_summary(rungs, rows, predictor)

## Pitfall on the hardest rung

Pitfall: optimizing the raw term and forgetting the cost. The wrong check looks only at the raw D5 metric; the fix restores the lesson's cost and gap before selecting a winner.

In [ ]:
name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
main_preds = predictor(x_tr, y_tr, x_te)
base_preds = logistic_baseline(x_tr, y_tr, x_te)
main_acc = accuracy_score(y_te, main_preds)
base_acc = accuracy_score(y_te, base_preds)
lesson = generalized_linear_models_method()
raw_only = 1.0 - main_acc
full_score = raw_only + lesson["cost"]
alt_score = (1.0 - base_acc) + lesson["alternative"]
print(f"Raw-only D5 error={raw_only:.3f} baseline_error={1.0 - base_acc:.3f}")
print(f"Full score with lesson cost={full_score:.3f} alternative score={alt_score:.3f}")
print(f"Lesson cost={lesson['cost']:.3f} gap={lesson['gap']:.3f}")
print(f"Macro-F1 sanity={f1_score(y_te, main_preds, average='macro'):.3f}")
assert lesson["gap"] > 0.0
assert 0.0 <= full_score

## Evaluate it + Practice

- Compare the headline metric with the no-skill or logistic baseline before claiming improvement.
- Run a cheap sanity check: shuffled labels should damage accuracy, and injected outliers should make robust regression matter.
- Ablation: turn off the key idea, such as Huber clipping, softmax normalization, the GLM link, covariance modeling, or Bayes priors; the D5 metric should usually drop.
- Failure signals include unstable validation gaps, wildly different scales, singular covariance warnings, or a D5 score that only wins before cost is included.

Practice 1: change the cost term and recompute the decision score.

Practice 2: rerun D5 after removing one informative feature group and compare the metric.

Practice 3: create a shuffled-label baseline and explain why it should fail.